<a href="https://colab.research.google.com/github/edwardoughton/IGARSS26/blob/main/notebook_2_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ IGARSS 2026 Summer School — Part 2 of 3 🛰️
## AI Refusnik to AI Evangelist: Excelling at AI-Assisted Coding for Satellite Image Analysis

**Instructor:** Ed Oughton, George Mason University

**Session duration:** ~60 minutes

**AI usage statement:** AI was used to support notebook code generation, but all content is based on my original GMU GGS course (https://github.com/edwardoughton/satellite-image-analysis/)

---

Welcome to Part 2!

In this notebook we will explore how to use **AI coding agents** to:

* Understand what AI agents are and how they differ from simple LLMs
* Use AI to generate and iteratively improve satellite image processing pipelines (testing/validation comes in part 3)
* Build a **land cover classification pipeline** driven by an AI agent
* Apply AI-assisted **change detection** between two time periods
* Understand prompt engineering patterns that work well for scientific coding tasks

> **Key message:** AI agents are most powerful when you are the expert who understands *what* you want. The AI handles the *how*. Importantly, we are not relinquishing the scientific process and our own critical thinking skills to AI... we are enabling better science by using AI.   

## 📗 Learning Objectives 📗

By the end of this notebook, you should be able to:

* Define how agentic AI modes go beyond basic prompting of an LLM
* Apply effective prompt engineering strategies for scientific coding tasks
* Use the OpenAI API (or compatible endpoint) in an agentic loop for iterative code generation
* Implement a **K-means land cover classification** pipeline with AI-generated code
* Implement a **NDVI change detection** workflow between two dates
* Critically evaluate AI-generated code, so what to trust and what to verify
* Understand the **"AI Evangelist" mindset** (by this I mean embracing AI as a research amplifier, not a replacement)

---

## The AI Refusnik to AI Evangelist Journey

Many scientists initially resist AI coding tools. Common reasons include:

- *"I don't trust code I didn't write myself."*
- *"AI hallucinations will corrupt my results."*
- *"It feels like cheating."*
- *"My domain is too specialized for generic AI tools."*

These are **valid concerns** — but they are concerns to be *managed*, not reasons to *avoid* AI.

The **AI Evangelist** position is:

> AI coding agents are **the most powerful productivity amplifier** since the invention of the scientific programming language. The question is not *whether* to use them, it's *how* to use them rigorously.

The keys to rigorous AI use in research are:

1. **You remain the domain expert** — AI writes code, you validate science
2. **Always test AI-generated code** — Part 3 of this course covers testing
3. **Prompt engineering is a skill** — learn to write precise prompts with strong context
4. **Iterate with AI** — treat it as a collaborator, not an oracle
5. **Cite and document** — be transparent about AI assistance in your workflow

---

## 1. Install and import dependencies

In [ ]:
!pip -q install openai pystac-client planetary-computer odc-stac rasterio numpy matplotlib scikit-learn scipy geopandas shapely

In [ ]:
import os
import json
import warnings
import textwrap
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.plot import show

import pystac_client
import planetary_computer
import odc.stac

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy import ndimage

warnings.filterwarnings('ignore')

DATA_DIR = Path('igarss26_data')
DATA_DIR.mkdir(exist_ok=True)

print('All packages imported.')

---

## 2. Setting up an AI Agent connection

We will use the **OpenAI Python client** to interact with an LLM. You can use:

- **OpenAI API** (requires API key from https://platform.openai.com/)
- **Azure OpenAI** (enterprise deployment)
- **GitHub Copilot API** (if you have Copilot Pro)
- **Ollama** (free, local, open-weight models like `llama3`, `codestral`)

For this session, we provide a pattern that works with any OpenAI-compatible endpoint. You can substitute your preferred provider.

> **Note:** If you do not have an API key, you can still follow along — we provide the AI-generated code outputs as pre-written cells below each exercise so you can compare.

In [ ]:
# Configuration: set your API key and model choice
# You can also set OPENAI_API_KEY as an environment variable

import os
from openai import OpenAI

# ─── Set your API key here or via environment variable ───
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# For GitHub Copilot API access:
# client = OpenAI(
#     base_url='https://models.inference.ai.azure.com',
#     api_key=os.environ.get('GITHUB_TOKEN')
# )
# MODEL = 'gpt-4o'

# Default OpenAI setup:
API_KEY = os.environ.get('OPENAI_API_KEY', '')
MODEL = 'gpt-4o-mini'  # cheaper, still capable for coding

if API_KEY:
    client = OpenAI(api_key=API_KEY)
    print(f'OpenAI client configured. Model: {MODEL}')
else:
    client = None
    print('No API key set. Pre-written code blocks below will serve as the AI outputs.')
    print('To enable AI generation: set OPENAI_API_KEY in Secrets (Colab) or environment.')

In [ ]:
def ask_ai_agent(prompt, system_prompt=None, model=MODEL, temperature=0.2):
    """
    Send a prompt to the AI agent and return the response text.

    Parameters
    ----------
    prompt : str
        The user message / task description.
    system_prompt : str, optional
        A system-level instruction that shapes the agent's behavior.
    model : str
        The model identifier.
    temperature : float
        Sampling temperature (0 = deterministic, 1 = creative).
        For code generation, low values (0.0–0.3) are preferred.

    Returns
    -------
    str
        The AI response text.
    """
    if client is None:
        return '[AI agent not configured — see pre-written code below]'

    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


# System prompt we use throughout this session
SCIENCE_SYSTEM_PROMPT = """
You are an expert geospatial scientist and Python developer specializing in
satellite image analysis, remote sensing, and Earth observation.

When writing code:
- Use numpy, rasterio, matplotlib, and scikit-learn unless otherwise specified
- Always include docstrings for functions
- Add inline comments explaining the scientific rationale, not just the syntax
- Validate inputs and handle nodata/NaN values explicitly
- Return code that is self-contained and executable
- Prefer scientific accuracy over brevity
""".strip()

print('AI agent helper function defined.')

---

## 3. Prompt Engineering for Scientific Code

Before building the pipeline, let's examine **what makes a good AI prompt** for scientific coding.

### The CAPE framework for scientific prompts:

| Element | Description | Example |
|---|---|---|
| **C**ontext | Describe your data and scientific domain | *"I have a 2D numpy array of Landsat NDVI values (float32, range -1 to 1, NaN for nodata)"* |
| **A**ction | State precisely what you want the code to do | *"Classify the NDVI array into 5 land cover classes using K-means"* |
| **P**arameters | Specify constraints, preferences, edge cases | *"Use sklearn, standardize features, set random_state=42, handle NaN by masking"* |
| **E**xpected output | Describe the expected result format | *"Return a 2D integer array with class labels 0–4 and a legend dictionary"* |

Let's compare a weak vs strong prompt:

In [ ]:
# WEAK PROMPT — vague, no context
weak_prompt = "Write Python code to classify a satellite image."

# STRONG PROMPT — using CAPE framework
strong_prompt = """
Context:
  I have two 2D numpy arrays from Landsat Collection 2 Level 2:
  - `ndvi`: float32, range [-1, 1], NaN for cloud/nodata pixels
  - `ndbi`: float32, range [-1, 1], same shape and nodata mask as ndvi
  Both arrays have shape (rows, cols) and represent a single summer scene
  over a mixed urban-suburban-vegetated landscape.

Action:
  Perform an unsupervised K-means land cover classification using these two
  spectral indices as input features. I want 5 classes.

Parameters:
  - Use sklearn.cluster.KMeans with n_clusters=5, random_state=42, n_init=10
  - Standardize features with sklearn.preprocessing.StandardScaler
  - Create a binary valid_mask to exclude NaN pixels from clustering
  - After clustering, sort classes by mean NDVI so class 0 = lowest NDVI
    (most built-up / bare), class 4 = highest NDVI (densest vegetation)

Expected output:
  - A 2D integer array `classified` (same shape as input), values 0–4
    for valid pixels, -1 for nodata pixels
  - A dictionary `class_labels` mapping class index to descriptive name
    (e.g., {0: 'Water/Built-up', ..., 4: 'Dense Vegetation'})
  - Print the number of pixels in each class
""".strip()

print('Strong prompt (CAPE framework):')
print('=' * 60)
print(strong_prompt)

In [ ]:
# Send the strong prompt to the AI agent
print('Sending prompt to AI agent...\n')
classification_code = ask_ai_agent(strong_prompt, system_prompt=SCIENCE_SYSTEM_PROMPT)
print('AI Agent Response:')
print('=' * 60)
print(classification_code)

In [ ]:
import numpy as np
import rasterio
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def classify_land_cover(ndvi, ndbi):
    """
    Classify land cover using NDVI and NDBI with K-means clustering.

    Parameters:
    - ndvi: 2D numpy array of NDVI values (float32), shape (rows, cols)
    - ndbi: 2D numpy array of NDBI values (float32), shape (rows, cols)

    Returns:
    - classified: 2D integer array of classified land cover (0-4 for valid pixels, -1 for nodata)
    - class_labels: dictionary mapping class index to descriptive name
    """
    # Validate input shapes
    if ndvi.shape != ndbi.shape:
        raise ValueError("NDVI and NDBI arrays must have the same shape.")

    # Create a mask for valid (non-NaN) pixels
    valid_mask = ~np.isnan(ndvi) & ~np.isnan(ndbi)

    # Extract valid NDVI and NDBI values
    valid_ndvi = ndvi[valid_mask].reshape(-1, 1)
    valid_ndbi = ndbi[valid_mask].reshape(-1, 1)

    # Stack NDVI and NDBI for clustering
    features = np.hstack((valid_ndvi, valid_ndbi))

    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Perform K-means clustering
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    kmeans.fit(features_scaled)

    # Create an empty classified array with nodata values
    classified = np.full(ndvi.shape, -1, dtype=int)

    # Assign cluster labels to valid pixels
    classified[valid_mask] = kmeans.labels_

    # Sort classes by mean NDVI
    class_means = [np.mean(ndvi[classified == i]) for i in range(5)]
    sorted_classes = np.argsort(class_means)

    # Remap classes to ensure class 0 = lowest NDVI and class 4 = highest NDVI
    for new_class, old_class in enumerate(sorted_classes):
        classified[classified == old_class] = new_class

    # Create class labels
    class_labels = {
        0: 'Water/Built-up',
        1: 'Low Vegetation',
        2: 'Medium Vegetation',
        3: 'High Vegetation',
        4: 'Dense Vegetation'
    }

    # Print the number of pixels in each class
    unique, counts = np.unique(classified, return_counts=True)
    pixel_counts = dict(zip(unique, counts))
    for class_index in range(5):
        print(f"Class {class_index} ({class_labels[class_index]}): {pixel_counts.get(class_index, 0)} pixels")

    return classified, class_labels


---

## 4. K-Means Land Cover Classification Pipeline

Now we will load the Landsat data from Part 1 and build a complete classification pipeline. The code below represents a **reviewed and validated version** of what the AI agent produced (with minor corrections).

This is the key workflow:

```
[Raw bands] → [Compute indices] → [Stack features] → [Mask nodata]
          → [Standardize] → [K-Means] → [Sort classes] → [Visualize]
```

In [ ]:
# Step 1: Reload Landsat data (or load from saved GeoTIFFs if you have them from Part 1)
# For demonstration we reconnect to Planetary Computer

STAC_API_URL = 'https://planetarycomputer.microsoft.com/api/stac/v1'
AOI_BBOX = [-77.20, 38.75, -76.90, 39.05]
TIME_RANGE = '2024-06-01/2024-09-30'
MAX_CLOUD = 15

catalog = pystac_client.Client.open(
    STAC_API_URL,
    modifier=planetary_computer.sign_inplace
)

landsat_search = catalog.search(
    collections=['landsat-c2-l2'],
    bbox=AOI_BBOX,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': MAX_CLOUD}},
)
landsat_items = list(landsat_search.items())
best_item = sorted(landsat_items, key=lambda x: x.properties.get('eo:cloud_cover', 100))[0]

LANDSAT_BANDS = ['red', 'green', 'blue', 'nir08', 'swir16']
landsat_ds = odc.stac.load(
    [best_item],
    bands=LANDSAT_BANDS,
    bbox=AOI_BBOX,
    resolution=30,
    groupby='solar_day'
)

ls_scene = landsat_ds.isel(time=0)

scale = 10000.0
red   = np.clip(ls_scene['red'].values.astype(float)   / scale, 0, 1)
green = np.clip(ls_scene['green'].values.astype(float) / scale, 0, 1)
blue  = np.clip(ls_scene['blue'].values.astype(float)  / scale, 0, 1)
nir   = np.clip(ls_scene['nir08'].values.astype(float) / scale, 0, 1)
swir1 = np.clip(ls_scene['swir16'].values.astype(float)/ scale, 0, 1)

print(f'Landsat scene loaded: {red.shape}')

In [ ]:
def safe_index(a, b):
    """Compute (a-b)/(a+b) normalized difference, returning NaN where a+b==0."""
    denom = a + b
    return np.where(denom == 0, np.nan, (a - b) / denom)


def kmeans_land_cover(
    ndvi: np.ndarray,
    ndwi: np.ndarray,
    ndbi: np.ndarray,
    n_clusters: int = 5,
    random_state: int = 42
):
    """
    Unsupervised K-means land cover classification using spectral indices.

    Pixels with NaN in any index are excluded from clustering and returned
    as -1 (nodata) in the output classification map.

    Classes are sorted by ascending mean NDVI so that:
      0 = lowest NDVI (water/built-up)
      n_clusters-1 = highest NDVI (dense vegetation)

    Parameters
    ----------
    ndvi, ndwi, ndbi : np.ndarray
        2D float arrays of spectral indices, shape (rows, cols).
    n_clusters : int
        Number of K-means clusters.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    classified : np.ndarray
        2D int array (rows, cols), values in [0, n_clusters-1] for valid
        pixels and -1 for nodata.
    class_labels : dict
        Mapping from class index to descriptive land cover label.
    cluster_stats : dict
        Per-class mean NDVI, NDWI, NDBI and pixel count.
    """
    rows, cols = ndvi.shape

    # Build valid pixel mask: all three indices must be finite
    valid_mask = (
        np.isfinite(ndvi) &
        np.isfinite(ndwi) &
        np.isfinite(ndbi)
    )

    # Stack features: shape (n_valid_pixels, 3)
    features = np.column_stack([
        ndvi[valid_mask],
        ndwi[valid_mask],
        ndbi[valid_mask]
    ])

    # Standardize: KMeans is sensitive to feature scale
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Run KMeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    raw_labels = kmeans.fit_predict(features_scaled)

    # Sort clusters by mean NDVI (ascending) for interpretable labels
    cluster_mean_ndvi = [
        features[raw_labels == k, 0].mean() for k in range(n_clusters)
    ]
    sorted_clusters = np.argsort(cluster_mean_ndvi)
    label_remap = {old: new for new, old in enumerate(sorted_clusters)}
    sorted_labels = np.vectorize(label_remap.get)(raw_labels)

    # Build output map: -1 for nodata
    classified = np.full((rows, cols), -1, dtype=np.int16)
    classified[valid_mask] = sorted_labels

    # Assign descriptive labels
    label_names = {
        0: 'Water / Low reflectance',
        1: 'Bare soil / Built-up dense',
        2: 'Urban / Mixed',
        3: 'Sparse vegetation',
        4: 'Dense vegetation'
    }
    class_labels = {k: label_names.get(k, f'Class {k}') for k in range(n_clusters)}

    # Compute per-class statistics
    cluster_stats = {}
    for k in range(n_clusters):
        mask_k = classified == k
        cluster_stats[k] = {
            'label': class_labels[k],
            'n_pixels': int(mask_k.sum()),
            'mean_ndvi': float(np.nanmean(ndvi[mask_k])),
            'mean_ndwi': float(np.nanmean(ndwi[mask_k])),
            'mean_ndbi': float(np.nanmean(ndbi[mask_k]))
        }

    return classified, class_labels, cluster_stats


# Compute indices
ndvi = safe_index(nir, red)
ndwi = safe_index(green, nir)
ndbi = safe_index(swir1, nir)

# Run classification
classified, class_labels, cluster_stats = kmeans_land_cover(ndvi, ndwi, ndbi)

print('Classification complete!')
print('\nClass statistics:')
for k, stats in cluster_stats.items():
    print(f"  Class {k} ({stats['label']}):")
    print(f"    Pixels: {stats['n_pixels']:,}")
    print(f"    Mean NDVI: {stats['mean_ndvi']:.3f}")
    print(f"    Mean NDWI: {stats['mean_ndwi']:.3f}")
    print(f"    Mean NDBI: {stats['mean_ndbi']:.3f}")

In [ ]:
# Visualize the land cover classification
COLORS = {
    -1: (0.5, 0.5, 0.5),   # Nodata — grey
     0: (0.1, 0.2, 0.8),   # Water — blue
     1: (0.8, 0.6, 0.3),   # Bare/built-up — tan
     2: (0.9, 0.8, 0.2),   # Urban/mixed — yellow
     3: (0.3, 0.8, 0.3),   # Sparse veg — light green
     4: (0.0, 0.5, 0.0),   # Dense veg — dark green
}

def classification_to_rgb(classified, colors):
    """Convert integer class map to RGB image for display."""
    rgb = np.zeros((*classified.shape, 3), dtype=float)
    for cls_val, color in colors.items():
        mask = classified == cls_val
        rgb[mask] = color
    return rgb

classified_rgb = classification_to_rgb(classified, COLORS)

# True-colour for reference
def stretch(arr, low=2, high=98):
    lo, hi = np.nanpercentile(arr, [low, high])
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

rgb = np.dstack([stretch(red), stretch(green), stretch(blue)])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(rgb)
axes[0].set_title('True-Colour Reference', fontsize=12)
axes[0].axis('off')

axes[1].imshow(classified_rgb)
axes[1].set_title('K-Means Land Cover Classification (5 classes)', fontsize=12)
axes[1].axis('off')

# Add legend
legend_labels = [class_labels.get(k, 'Nodata') for k in sorted(COLORS.keys()) if k >= 0]
legend_colors = [COLORS[k] for k in sorted(COLORS.keys()) if k >= 0]
patches = [mpatches.Patch(color=c, label=l) for c, l in zip(legend_colors, legend_labels)]
axes[1].legend(handles=patches, loc='lower right', fontsize=9, framealpha=0.85)

plt.suptitle('Unsupervised K-Means Land Cover Classification — Landsat', fontsize=13)
plt.tight_layout()
plt.show()

---

## 5. Multi-Temporal Change Detection with AI-Assisted Code

Change detection is one of the most impactful applications of multi-temporal satellite imagery. Here we will:

1. Load a **second Landsat scene** from a different season (or year)
2. Use an AI agent to generate the change detection pipeline
3. Critically evaluate and clean up the AI-generated code
4. Visualize vegetation change using **NDVI differencing**

### Change Detection Approaches

| Method | Description | When to use |
|---|---|---|
| **Index differencing** | ΔIndex = Index_t2 − Index_t1 | Quick exploratory analysis |
| **Image differencing** | Per-band subtraction | Multi-band change analysis |
| **CVA (Change Vector Analysis)** | Magnitude + direction of spectral change | Full spectral change characterization |
| **Post-classification comparison** | Classify both dates, compare maps | Categorical land cover change |
| **Bi-temporal deep learning** | CNN/Transformer on image pairs | Complex change patterns |

In [ ]:
# Load a second Landsat scene — we use a different time period to see seasonal change
winter_search = catalog.search(
    collections=['landsat-c2-l2'],
    bbox=AOI_BBOX,
    datetime='2024-01-01/2024-04-30',  # winter/spring
    query={'eo:cloud_cover': {'lt': MAX_CLOUD}},
)
winter_items = list(winter_search.items())

if winter_items:
    best_winter = sorted(winter_items, key=lambda x: x.properties.get('eo:cloud_cover', 100))[0]
    print(f'Winter/spring scene: {best_winter.id}')
    print(f'Date: {best_winter.datetime}')
    print(f'Cloud cover: {best_winter.properties.get("eo:cloud_cover")}%')
else:
    print('No suitable winter scenes found. Adjust time range or cloud threshold.')

In [ ]:
# Load winter scene
winter_ds = odc.stac.load(
    [best_winter],
    bands=['red', 'green', 'blue', 'nir08', 'swir16'],
    bbox=AOI_BBOX,
    resolution=30,
    groupby='solar_day'
)

w_scene = winter_ds.isel(time=0)

LANDSAT_SCALE = 0.0000275
LANDSAT_OFFSET = -0.2

w_red  = np.clip(w_scene['red'].values.astype(float) * LANDSAT_SCALE + LANDSAT_OFFSET, 0, 1)
w_green= np.clip(w_scene['green'].values.astype(float) * LANDSAT_SCALE + LANDSAT_OFFSET, 0, 1)
w_nir  = np.clip(w_scene['nir08'].values.astype(float) * LANDSAT_SCALE + LANDSAT_OFFSET, 0, 1)

ndvi_winter = safe_index(w_nir, w_red)

print(f'Winter NDVI — mean: {np.nanmean(ndvi_winter):.3f}')
print(f'Summer NDVI — mean: {np.nanmean(ndvi):.3f}')

In [ ]:
# AI-assisted change detection prompt
change_prompt = """
Context:
  I have two 2D numpy float arrays of NDVI computed from Landsat Collection 2 Level 2:
  - `ndvi_t1`: winter/spring season NDVI, shape (rows, cols), NaN for nodata
  - `ndvi_t2`: summer season NDVI, same shape and nodata convention
  Both represent the same geographic extent at 30 m resolution.

Action:
  Implement an NDVI difference change detection workflow that:
  1. Computes delta_ndvi = ndvi_t2 - ndvi_t1 (positive = greening, negative = browning)
  2. Creates a binary change mask using a threshold of ±0.15
  3. Applies a 3×3 median filter to remove salt-and-pepper noise
  4. Classifies pixels as: 'significant greening', 'no significant change', 'significant browning'

Parameters:
  - Only consider pixels where both t1 and t2 are finite (not NaN)
  - Use scipy.ndimage.median_filter for spatial smoothing
  - Threshold: change > +0.15 = greening, change < -0.15 = browning
  - Return a 2D integer array: 1=greening, 0=no change, -1=browning, NaN=nodata

Expected output:
  - A function `detect_ndvi_change(ndvi_t1, ndvi_t2, threshold=0.15)` that returns
    (delta_ndvi, change_class) where change_class is an integer array (-1, 0, 1)
  - Print summary: percentage of pixels in each class
""".strip()

print('Sending change detection prompt to AI agent...\n')
change_code = ask_ai_agent(change_prompt, system_prompt=SCIENCE_SYSTEM_PROMPT)
print('AI Agent Response:')
print('=' * 60)
print(change_code)

In [ ]:
# Validated implementation of the AI-generated change detection function

def detect_ndvi_change(ndvi_t1: np.ndarray, ndvi_t2: np.ndarray, threshold: float = 0.15):
    """
    Detect significant NDVI change between two time periods.

    Pixels are classified as:
      +1 = significant greening (delta_ndvi > +threshold)
       0 = no significant change (|delta_ndvi| <= threshold)
      -1 = significant browning (delta_ndvi < -threshold)
    Nodata (NaN in either input) → set to np.nan in delta_ndvi and
    a sentinel value of -9 in change_class for easy masking.

    A 3×3 median filter is applied to the delta_ndvi array before
    thresholding to reduce salt-and-pepper noise.

    Parameters
    ----------
    ndvi_t1, ndvi_t2 : np.ndarray
        2D float arrays of NDVI, same shape, NaN for nodata.
    threshold : float
        Absolute NDVI change threshold.

    Returns
    -------
    delta_ndvi : np.ndarray
        Smoothed NDVI difference (t2 - t1). NaN where nodata.
    change_class : np.ndarray
        Integer class array (-1, 0, 1) for valid pixels, -9 for nodata.
    """
    # Valid pixel mask: both arrays must be finite
    valid = np.isfinite(ndvi_t1) & np.isfinite(ndvi_t2)

    # Compute raw difference
    delta = np.full_like(ndvi_t1, np.nan)
    delta[valid] = ndvi_t2[valid] - ndvi_t1[valid]

    # Spatial smoothing: fill NaN with 0 before filter, then reapply mask
    delta_filled = np.where(valid, delta, 0)
    delta_smooth = ndimage.median_filter(delta_filled, size=3)
    delta_smooth = np.where(valid, delta_smooth, np.nan)

    # Classify change
    change_class = np.full_like(delta_smooth, -9, dtype=np.int8)
    change_class[valid & (delta_smooth >  threshold)] =  1   # greening
    change_class[valid & (delta_smooth < -threshold)] = -1   # browning
    change_class[valid & (np.abs(delta_smooth) <= threshold)] = 0  # stable

    # Print summary
    n_valid = valid.sum()
    n_green = (change_class == 1).sum()
    n_brown = (change_class == -1).sum()
    n_stable= (change_class == 0).sum()
    print(f'Change detection summary (threshold = ±{threshold}):')
    print(f'  Greening : {n_green:,} pixels ({100*n_green/n_valid:.1f}%)')
    print(f'  Stable   : {n_stable:,} pixels ({100*n_stable/n_valid:.1f}%)')
    print(f'  Browning : {n_brown:,} pixels ({100*n_brown/n_valid:.1f}%)')

    return delta_smooth, change_class


delta_ndvi, change_class = detect_ndvi_change(ndvi_winter, ndvi)

In [ ]:
# Visualize change detection results
valid_mask = change_class != -9
delta_display = np.where(valid_mask, delta_ndvi, np.nan)

change_rgb = np.zeros((*change_class.shape, 3))
change_rgb[change_class ==  1] = [0.0, 0.6, 0.0]   # green = greening
change_rgb[change_class ==  0] = [0.9, 0.9, 0.9]   # grey  = stable
change_rgb[change_class == -1] = [0.8, 0.1, 0.1]   # red   = browning
change_rgb[change_class == -9] = [0.5, 0.5, 0.5]   # dark  = nodata

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(delta_display, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
axes[0].set_title('ΔNDVI (Summer − Winter/Spring)\nPositive = more green', fontsize=11)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

axes[1].imshow(change_rgb)
axes[1].set_title('Change Classification\n(|ΔNDVI| > 0.15)', fontsize=11)
axes[1].axis('off')
patches_legend = [
    mpatches.Patch(color='green', label='Greening'),
    mpatches.Patch(color='lightgrey', label='Stable'),
    mpatches.Patch(color='red', label='Browning'),
]
axes[1].legend(handles=patches_legend, loc='lower right', fontsize=9)

# NDVI distributions for both dates
axes[2].hist(ndvi_winter[np.isfinite(ndvi_winter)].ravel(), bins=80,
             alpha=0.6, color='royalblue', label='Winter/Spring NDVI', density=True)
axes[2].hist(ndvi[np.isfinite(ndvi)].ravel(), bins=80,
             alpha=0.6, color='forestgreen', label='Summer NDVI', density=True)
axes[2].set_xlabel('NDVI')
axes[2].set_ylabel('Density')
axes[2].set_title('NDVI Distribution\nWinter/Spring vs Summer', fontsize=11)
axes[2].legend(fontsize=9)

plt.suptitle('NDVI Change Detection: Winter/Spring → Summer', fontsize=13)
plt.tight_layout()
plt.show()

### ✏️ Exercise 5.1 — Agentic Iteration

The current change detection only uses NDVI. However, we also have NDBI (built-up index). A significant decrease in NDVI **combined with** an increase in NDBI could indicate **urban expansion** (conversion of vegetation to built-up land).

**Your task:**

1. Write a CAPE-format prompt asking the AI agent to extend `detect_ndvi_change` to also use NDBI change
2. Add a new class: `'Possible urban expansion'` (ΔNDVI < -0.15 AND ΔNDBI > 0.1)
3. Send the prompt to the AI agent (`ask_ai_agent(...)`)
4. Evaluate the AI response: What would you need to change or fix?
5. Implement the revised function and visualize the result

*This exercise illustrates the **iterative** nature of AI-assisted coding: the AI provides a first draft, you refine it as the domain expert.*

In [ ]:
# Your CAPE prompt
my_prompt = """
Context: ...
Action: ...
Parameters: ...
Expected output: ...
"""

# response = ask_ai_agent(my_prompt, system_prompt=SCIENCE_SYSTEM_PROMPT)
# print(response)

---

## 6. Critically Evaluating AI-Generated Code

The most important skill for an AI Evangelist is **knowing when to trust AI output and when to push back**.

### Common failure modes for AI geospatial code:

| Failure mode | Example | How to catch it |
|---|---|---|
| **Wrong scaling** | Forgetting to divide Landsat values by 10000 | Check value range after loading |
| **CRS mismatch** | Comparing arrays from different projections | Always check `.crs` attributes |
| **NaN propagation** | Not masking clouds before statistics | Check output for unexpected NaN patterns |
| **Wrong band name** | Using 'B4' vs 'red' for the same band | Read STAC item `assets` dictionary |
| **Off-by-one errors** | Image shape vs. coordinate array length | Print shapes at each step |
| **Silent exceptions** | `try/except` that swallows errors | Use specific exception types |

### A simple code review checklist for AI-generated geospatial code:

1. ☑ Are all input arrays the same shape before operations?
2. ☑ Are reflectance values in the expected range [0, 1]?
3. ☑ Is the CRS consistent throughout?
4. ☑ Are nodata/cloud pixels properly masked before statistics?
5. ☑ Does the output make physical sense (e.g., NDVI > 0.6 in forests, < 0.1 in water)?

In [ ]:
def validate_spectral_array(arr, name, expected_range=(-1, 1)):
    """
    Quick sanity check for spectral index arrays.
    Prints warnings if values fall outside expected bounds.
    """
    valid = arr[np.isfinite(arr)]
    if len(valid) == 0:
        print(f'  ⚠️  {name}: ALL values are NaN!')
        return

    lo, hi = valid.min(), valid.max()
    pct_nan = 100 * np.isnan(arr).mean()

    status = '✅' if expected_range[0] <= lo and hi <= expected_range[1] else '⚠️ '
    print(f'  {status} {name}: range=[{lo:.3f}, {hi:.3f}]  '
          f'mean={valid.mean():.3f}  NaN={pct_nan:.1f}%')

    if lo < expected_range[0] - 0.05 or hi > expected_range[1] + 0.05:
        print(f'       → Expected range {expected_range}. Check scaling!')


print('Validation of spectral arrays:')
validate_spectral_array(ndvi, 'NDVI (summer)')
validate_spectral_array(ndwi, 'NDWI (summer)')
validate_spectral_array(ndbi, 'NDBI (summer)')
validate_spectral_array(ndvi_winter, 'NDVI (winter)')
validate_spectral_array(delta_ndvi, 'ΔNDVI', expected_range=(-2, 2))
validate_spectral_array(red, 'Red reflectance', expected_range=(0, 1))
validate_spectral_array(nir, 'NIR reflectance', expected_range=(0, 1))

---

## ✅ Part 2 Summary

In this notebook you have:

- ✅ Understood the difference between LLM chat, code completion, and **agentic** AI modes
- ✅ Applied the **CAPE framework** for writing scientific AI prompts
- ✅ Used `ask_ai_agent()` to generate a land cover classification pipeline
- ✅ Implemented a **K-Means unsupervised land cover classification** from Landsat spectral indices
- ✅ Built an **NDVI change detection** workflow for multi-temporal analysis
- ✅ Developed a **validation checklist** and helper functions to catch AI code errors

### 🔜 Up next: Part 3

In **Part 3**, we will formalize the validation process: writing **unit tests**, **accuracy assessments**, and **uncertainty quantification** for the pipelines we have built.

**Continue to `notebook_3_testing_validation.ipynb`** →